# Caso de estudio: Segmentación del consumo de café en el trabajo con K-Means

## Objetivo de aprendizaje
En este ejercicio aplicaremos un **modelo descriptivo** sobre el consumo de café en el trabajo, utilizando **K-Means** para agrupar trabajadores con comportamientos similares.

Al finalizar, el estudiante debería ser capaz de:

- comprender la diferencia entre un problema **descriptivo** y uno **predictivo**
- cargar un dataset en Google Colab
- analizar variables de consumo de café
- preparar y escalar datos
- aplicar **K-Means**
- seleccionar un número adecuado de clusters
- evaluar si el modelo está operando correctamente con métricas apropiadas
- interpretar los grupos encontrados

---

## Contexto del caso
Una empresa desea comprender los **patrones de consumo de café entre sus trabajadores** para mejorar sus espacios de descanso, apoyar iniciativas de bienestar laboral y entender hábitos asociados al estrés, el sueño y la carga de trabajo.

Para ello dispone de variables como:

- edad
- horas de trabajo al día
- tazas de café al día
- horas de sueño
- nivel de estrés
- cantidad de reuniones diarias
- trabajo remoto

A diferencia de un modelo predictivo, aquí **no existe una variable objetivo a predecir**.

El propósito es:

> **descubrir grupos de trabajadores con características similares respecto al consumo de café y su contexto laboral**

Por ello estamos frente a un **modelo descriptivo**.

---

## ¿Por qué usar K-Means?
K-Means es una técnica de agrupamiento que permite:

- identificar grupos similares dentro de los datos
- resumir perfiles de comportamiento
- segmentar observaciones sin etiquetas previas
- facilitar la interpretación de patrones ocultos

En este caso, nos ayudará a responder preguntas como:

- ¿existen trabajadores con alto consumo de café y alto estrés?
- ¿hay perfiles de bajo consumo y mejor descanso?
- ¿se distinguen grupos por reuniones, sueño o trabajo remoto?


## Paso 1: Cargar el dataset

En Google Colab primero debemos subir el archivo CSV.

Ejecuta la siguiente celda y selecciona el archivo:

**dataset_consumo_cafe_trabajo.csv**


In [ ]:
from google.colab import files
uploaded = files.upload()

## Paso 2: Importar librerías

Estas librerías nos permitirán:

- manipular datos con **pandas**
- hacer cálculos con **numpy**
- escalar variables con **StandardScaler**
- agrupar con **KMeans**
- evaluar la calidad del agrupamiento
- visualizar resultados con **matplotlib**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

## Paso 3: Leer el archivo CSV

Ahora cargamos el dataset y mostramos sus primeras filas para verificar que los datos se hayan leído correctamente.


In [ ]:
df = pd.read_csv("dataset_consumo_cafe_trabajo.csv")

print("Dimensión del dataset:", df.shape)
df.head()

## Paso 4: Comprender las variables del problema

En este caso trabajaremos con las siguientes variables:

- **EDAD**: edad del trabajador
- **HORAS_TRABAJO_DIA**: horas de trabajo al día
- **TAZAS_CAFE_DIA**: cantidad de tazas de café consumidas al día
- **HORAS_SUENO**: horas promedio de sueño
- **NIVEL_ESTRES**: nivel de estrés reportado
- **REUNIONES_DIA**: cantidad de reuniones por día
- **TRABAJO_REMOTO**: si trabaja remoto o no

### Importante
Aquí no existe una variable objetivo como en los problemas predictivos.  
Todas las variables se analizan al mismo nivel, porque el propósito es encontrar **patrones ocultos**.


In [ ]:
print("Columnas del dataset:")
for c in df.columns:
    print("-", c)

## Paso 5: Revisión general de los datos

Antes de aplicar el modelo debemos revisar:

- tipos de datos
- valores nulos
- estadísticas generales

Esto permite verificar que el dataset esté correctamente preparado.


In [ ]:
print("Información general:")
print(df.info())

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nResumen estadístico:")
df.describe()

## Paso 6: Exploración inicial de los datos

En un modelo descriptivo, la exploración inicial ayuda a comprender el comportamiento general de las variables.

Observaremos algunas relaciones asociadas al consumo de café.


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["TAZAS_CAFE_DIA"], bins=6)
plt.title("Distribución de tazas de café por día")
plt.xlabel("Tazas de café")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["NIVEL_ESTRES"], df["TAZAS_CAFE_DIA"])
plt.title("Nivel de estrés vs tazas de café")
plt.xlabel("Nivel de estrés")
plt.ylabel("Tazas de café al día")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["HORAS_SUENO"], df["TAZAS_CAFE_DIA"])
plt.title("Horas de sueño vs tazas de café")
plt.xlabel("Horas de sueño")
plt.ylabel("Tazas de café al día")
plt.show()

## Paso 7: Preparar los datos para K-Means

K-Means utiliza distancias para formar grupos.  
Por eso, si las variables están en escalas muy distintas, el modelo puede dar demasiado peso a algunas columnas.

Aplicaremos **estandarización** para dejar todas las variables en una escala comparable:

- media cercana a 0
- desviación estándar cercana a 1


In [ ]:
X = df.copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Forma original:", X.shape)
print("Forma escalada:", X_scaled.shape)

## Paso 8: ¿Cómo funciona K-Means?

K-Means busca formar grupos de manera que:

- los elementos de un mismo cluster sean parecidos entre sí
- los elementos de clusters distintos sean diferentes

### Idea general del algoritmo
1. Se define un número de clusters (**k**)
2. Se crean centros iniciales
3. Cada observación se asigna al centro más cercano
4. Los centros se recalculan
5. El proceso se repite hasta estabilizarse

Por eso una decisión importante en K-Means es elegir un valor adecuado para **k**.


## Paso 9: Método del codo para elegir el número de clusters

Una forma clásica de elegir **k** es observar la **inercia**.

### ¿Qué es la inercia?
La inercia mide qué tan compactos son los grupos.

- mientras menor, mejor
- pero si seguimos aumentando k, siempre tenderá a bajar

Por eso observamos el punto donde la mejora deja de ser tan grande:  
ese punto suele llamarse **codo**.


In [ ]:
inertia = []

for k in range(1, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(1,10), inertia, marker='o')
plt.title("Método del codo")
plt.xlabel("Número de clusters")
plt.ylabel("Inercia")
plt.show()

## Paso 10: Elegir un valor de k y entrenar el modelo

En este ejemplo trabajaremos con **3 clusters**.

Este valor debe justificarse observando el método del codo y luego validarse con métricas adicionales.


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df["CLUSTER"] = clusters

print("Modelo K-Means entrenado correctamente.")
df.head()

## Paso 11: Revisar los centroides del modelo

Los centroides representan el “centro” de cada grupo en el espacio escalado.

No siempre son fáciles de interpretar directamente, pero ayudan a entender cómo el modelo organiza los datos.


In [ ]:
centroides = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=X.columns
)

centroides

## Paso 12: Métricas para determinar si K-Means está bien aplicado

En modelos descriptivos no usamos accuracy, recall o matriz de confusión.

En cambio, para evaluar clustering utilizamos métricas como:

### 1. Silhouette Score
Mide qué tan bien separados están los grupos.

- cercano a **1** → grupos bien definidos
- cercano a **0** → grupos mezclados
- negativo → mala agrupación

### 2. Davies-Bouldin Index
Mide la similitud entre clusters.

- mientras **menor**, mejor

### 3. Calinski-Harabasz Score
Evalúa separación entre clusters y cohesión interna.

- mientras **mayor**, mejor

Estas métricas permiten revisar si el modelo está operando correctamente y si el agrupamiento tiene sentido.


In [ ]:
sil_score = silhouette_score(X_scaled, clusters)
db_score = davies_bouldin_score(X_scaled, clusters)
ch_score = calinski_harabasz_score(X_scaled, clusters)

print("MÉTRICAS DEL MODELO K-MEANS")
print("---------------------------")
print("Silhouette Score       :", round(sil_score, 4))
print("Davies-Bouldin Index   :", round(db_score, 4))
print("Calinski-Harabasz Score:", round(ch_score, 4))

## Paso 13: Interpretación guiada de las métricas

El siguiente bloque entrega una interpretación básica para apoyar la discusión en clase.


In [ ]:
print("INTERPRETACIÓN GENERAL")
print("----------------------")

if sil_score >= 0.50:
    print("Los clusters presentan una separación buena o alta.")
elif sil_score >= 0.25:
    print("Los clusters presentan una separación moderada o aceptable.")
else:
    print("Los clusters presentan una separación débil y podrían mejorarse.")

if db_score < 1:
    print("El índice Davies-Bouldin sugiere grupos relativamente compactos y diferenciados.")
else:
    print("El índice Davies-Bouldin sugiere revisar la calidad de separación entre clusters.")

print("Recordatorio:")
print("- En Silhouette, mientras más alto, mejor.")
print("- En Davies-Bouldin, mientras más bajo, mejor.")
print("- En Calinski-Harabasz, mientras más alto, mejor.")

## Paso 14: Caracterizar los clusters

Ahora calcularemos los promedios de cada variable por cluster.

Aquí ocurre la parte más importante del modelo descriptivo:

> **interpretar qué representa cada grupo**


In [ ]:
perfil_clusters = df.groupby("CLUSTER").mean(numeric_only=True)
perfil_clusters

## Paso 15: Visualizar los clusters

Para representar visualmente los grupos, utilizaremos dos variables relevantes:
- nivel de estrés
- tazas de café al día

Esto no muestra toda la complejidad del modelo, pero ayuda a interpretar mejor los perfiles.


In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(
    df["NIVEL_ESTRES"],
    df["TAZAS_CAFE_DIA"],
    c=df["CLUSTER"]
)
plt.title("Clusters según nivel de estrés y consumo de café")
plt.xlabel("Nivel de estrés")
plt.ylabel("Tazas de café al día")
plt.show()

## Paso 16: Tamaño de cada cluster

También es útil revisar cuántas observaciones quedaron en cada grupo.


In [ ]:
df["CLUSTER"].value_counts().sort_index()

## Paso 17: Ejemplo de interpretación de perfiles

Con la tabla de promedios puedes generar descripciones como:

- **Cluster 0**: trabajadores con mayor nivel de estrés y mayor consumo de café
- **Cluster 1**: trabajadores con menos café y más horas de sueño
- **Cluster 2**: trabajadores con patrón intermedio o asociado a reuniones y trabajo remoto

Estas etiquetas deben construirse observando los datos, no inventarse sin evidencia.


## Paso 18: Conclusión del ejercicio

### ¿Por qué esta técnica es adecuada?
K-Means es adecuado en este caso porque:

- el problema es **descriptivo**
- no existe variable objetivo
- queremos descubrir grupos ocultos
- el algoritmo permite segmentar observaciones similares
- las métricas ayudan a validar la calidad del agrupamiento

### ¿Cómo sabemos si el modelo está operando correctamente?
Lo evaluamos mediante:

- método del codo
- **silhouette score**
- **Davies-Bouldin index**
- **Calinski-Harabasz score**
- interpretación coherente de los clusters

### Idea final
En modelos descriptivos no preguntamos:

> “¿predijo correctamente?”

Sino:

> “¿agrupó de manera coherente y útil?”

Ese es el criterio correcto en este tipo de análisis.


## Preguntas para los estudiantes

1. ¿Por qué este caso corresponde a un modelo descriptivo?
2. ¿Qué función cumple K-Means en este ejercicio?
3. ¿Por qué es necesario escalar los datos antes de aplicar K-Means?
4. ¿Qué representa la inercia en el método del codo?
5. ¿Qué significa un silhouette score alto?
6. ¿Qué significa un Davies-Bouldin bajo?
7. ¿Qué características distinguen a cada cluster?
8. ¿Qué decisiones podría tomar la empresa a partir de estos grupos?
